In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../../data/bpic12.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:REG_DATE_HR": "string",
        "case:REG_DATE_DAY": "string",
        "case:REG_DATE_MON": "string",
        "case:AMOUNT_REQ": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
3,173688,2011-10-01 11:42:43.308,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
4,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
5,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
6,173688,2011-10-01 11:45:11.197,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
7,173688,2011-10-01 11:45:11.380,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
8,173688,2011-10-10 11:33:03.668,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
9,173688,2011-10-13 10:37:29.226,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AMOUNT_REQ', 'case:REG_DATE_DAY', 'case:REG_DATE_HR', 'case:REG_DATE_MON', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 1315.95]                          0.4980     quantile_derived    
case:AMOUNT_REQ                continuous     case     yes    [3000.00, 40000.00]                      5000.0000  quantile_derived    
case:REG_DATE_DAY              categorical    case     yes    ['Friday', 'Monday', 'Saturday', ...]    N/A        

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load(
    path = "../pretrained_models/"
)

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'A_FINALIZED', 'O_SELECTED'},
 {'A_DECLINED', 'O_DECLINED'},
 {'A_ACTIVATED', 'A_APPROVED', 'A_REGISTERED', 'O_ACCEPTED'}]

In [16]:
engine.branching_sets

[{'A_CANCELLED',
  'A_FINALIZED',
  'O_CREATED',
  'O_SELECTED',
  'O_SENT',
  'O_SENT_BACK'},
 {'A_ACTIVATED',
  'A_APPROVED',
  'A_DECLINED',
  'A_REGISTERED',
  'O_ACCEPTED',
  'O_DECLINED'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic12-cf_seed777_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,210128,3,1,0,0.068989,0.137977,0.0,0.250000,0.000000,...,0.318989,0.000000,0.068989,0.0,0.137977,0.250000,0.000000,0.000000,0.000000,0.000000
1,1,187256,3,1,0,0.004104,0.008207,0.0,0.250000,0.000000,...,0.330216,0.000000,0.004104,0.0,0.008207,0.250000,0.076112,0.076112,0.000000,0.000000
2,1,211510,3,1,0,0.062102,0.124204,0.0,0.250000,0.000000,...,0.540307,0.000000,0.062102,0.0,0.124204,0.250000,0.228205,0.228205,0.000000,0.000000
3,1,184360,3,1,0,0.090133,0.180267,0.0,0.250000,0.333333,...,1.205111,0.333333,0.090133,0.0,0.180267,0.250000,0.531644,0.531644,0.999989,0.999989
4,1,204757,3,1,0,0.081919,0.163839,0.0,0.250000,0.000000,...,0.633873,0.000000,0.081919,0.0,0.163839,0.250000,0.301954,0.301954,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,49,195482,14,1,2,0.113103,0.226205,0.0,0.392857,0.419355,...,1.902411,0.419355,0.113103,0.0,0.226205,0.392857,0.977096,0.693675,1.000000,0.999999
308,49,183292,14,1,2,0.128537,0.257073,0.0,0.428571,0.419355,...,1.940576,0.419355,0.128537,0.0,0.257073,0.428571,0.964113,0.740334,1.000000,0.999999
309,49,196861,14,1,2,0.172230,0.344460,0.0,0.428571,0.483871,...,2.044466,0.483871,0.172230,0.0,0.344460,0.428571,0.959794,0.564320,1.000000,0.999995
310,49,185452,14,1,2,0.112009,0.224018,0.0,0.392857,0.483871,...,1.944277,0.483871,0.112009,0.0,0.224018,0.392857,0.955539,0.726513,1.000000,0.999999


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,210128,3,1,0,0.068135,0.136270,0.000000,0.250000,0.000000,...,0.608404,0.000000,0.068135,0.000000,0.136270,0.250000,0.290269,0.290269,0.000000,0.000000
1,1,187256,3,1,0,0.067578,0.135156,0.000000,0.250000,0.000000,...,0.317578,0.000000,0.067578,0.000000,0.135156,0.250000,0.000000,0.000000,0.000000,0.000000
2,1,211510,3,1,0,0.011949,0.023898,0.000000,0.250000,0.000000,...,0.738773,0.000000,0.011949,0.000000,0.023898,0.250000,0.476824,0.476824,0.000000,0.000000
3,1,184360,3,1,0,0.336497,0.072995,0.600000,0.625000,0.000000,...,1.349043,0.000000,0.336497,0.600000,0.072995,0.625000,0.387546,0.387546,0.000000,0.000000
4,1,204757,3,1,0,0.123332,0.046664,0.200000,0.375000,0.000000,...,0.986046,0.000000,0.123332,0.200000,0.046664,0.375000,0.487715,0.487715,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,49,195482,14,1,2,0.347852,0.362370,0.333333,0.571429,0.161290,...,1.693165,0.161290,0.347852,0.333333,0.362370,0.571429,0.612594,0.612951,0.999997,0.999998
308,49,183292,14,1,2,0.284722,0.369444,0.200000,0.500000,0.419355,...,2.163817,0.419355,0.284722,0.200000,0.369444,0.500000,0.959740,0.000000,1.000000,0.000000
309,49,196861,14,1,2,0.245729,0.358125,0.133333,0.500000,0.483871,...,2.188772,0.483871,0.245729,0.133333,0.358125,0.500000,0.959171,0.000000,1.000000,0.000000
310,49,185452,14,1,2,0.361556,0.456445,0.266667,0.571429,0.483871,...,2.368445,0.483871,0.361556,0.266667,0.456445,0.571429,0.951590,0.000000,1.000000,0.000000


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()